# 04 — Fresh-process CPU scaling on the department desktop

This notebook measures the production six-link shift across increasing numbers
of physical CPU cores. It reuses the WDM coefficient cache created by notebook
03.

All worker records, summaries and plots are saved directly beside the notebook
with a `_local_wdm_` prefix.

In [ ]:
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

import WDM_LISAresponse


NOTEBOOK_DIR = Path.cwd().resolve()
ORBIT_FILE = NOTEBOOK_DIR / "esa-trailing-orbits.h5"

if not ORBIT_FILE.exists():
    raise FileNotFoundError(
        "Place esa-trailing-orbits.h5 in the same directory as these notebooks. "
        f"Current notebook directory: {NOTEBOOK_DIR}"
    )

if not hasattr(os, "sched_setaffinity"):
    raise RuntimeError(
        "This notebook requires Linux CPU affinity and is intended for "
        "the department desktop."
    )


def git_state(package_file: str) -> dict[str, object]:
    """Return the current Git commit and whether the checkout is dirty."""
    completed = subprocess.run(
        [
            "git",
            "-C",
            str(Path(package_file).resolve().parent),
            "rev-parse",
            "--show-toplevel",
        ],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        return {"root": None, "commit": None, "dirty": None}

    root = Path(completed.stdout.strip()).resolve()
    commit = subprocess.run(
        ["git", "-C", str(root), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    dirty = bool(
        subprocess.run(
            ["git", "-C", str(root), "status", "--porcelain"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    )
    return {"root": str(root), "commit": commit, "dirty": dirty}


print("Notebook directory:", NOTEBOOK_DIR)
print("Orbit file:", ORBIT_FILE)
print("Machine:", platform.node())
print("Platform:", platform.platform())
print("Source state:", git_state(WDM_LISAresponse.__file__))

In [ ]:
# Configuration must match notebook 03.

N = 3_200_000
dt = 10.0
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25
CALC_M0 = False

ROW_CHUNK_SIZE = 32_640
LAG_BLOCK_SIZE = 17

WARMUPS = 2
REPEATS = 9

theta = np.pi / 2
phi = 0.0

INPUT_CACHE = NOTEBOOK_DIR / "_local_wdm_03_wdm_coefficients.npz"
if not INPUT_CACHE.exists():
    raise FileNotFoundError(
        "Run 03_profile_one_arm_desktop.ipynb first. "
        f"Expected cache: {INPUT_CACHE}"
    )

WORKER_SCRIPT = NOTEBOOK_DIR / "_local_wdm_04_scaling_worker.py"
REFERENCE_OUTPUT = NOTEBOOK_DIR / "_local_wdm_04_one_core_reference.npy"

print("Input cache:", INPUT_CACHE)
print("Worker script:", WORKER_SCRIPT)

## 1. Detect one representative logical CPU per physical core

The `lscpu` parser selects one logical CPU for each `(socket, core)` pair.
The sweep uses compact prefixes of this list.


In [ ]:
completed = subprocess.run(
    ["lscpu", "-p=CPU,CORE,SOCKET"],
    capture_output=True,
    text=True,
    check=True,
)

representatives = {}
for line in completed.stdout.splitlines():
    if not line or line.startswith("#"):
        continue
    cpu, core, socket = (int(value) for value in line.split(","))
    representatives.setdefault((socket, core), cpu)

PHYSICAL_CPU_IDS = [
    representatives[key]
    for key in sorted(representatives)
]
physical_count = len(PHYSICAL_CPU_IDS)

candidate_counts = [1, 2, 4, 8, 12, 16, 20, 24, 32, physical_count]
CORE_COUNTS = sorted(
    {
        count
        for count in candidate_counts
        if 1 <= count <= physical_count
    }
)

print("Physical-core representatives:", PHYSICAL_CPU_IDS)
print("Core counts:", CORE_COUNTS)


## 2. Write the fresh-process worker


In [ ]:
WORKER_SOURCE = r"""
from __future__ import annotations

import argparse
import json
import os
import platform
import time
from pathlib import Path


parser = argparse.ArgumentParser()
parser.add_argument("--input", required=True)
parser.add_argument("--orbit-file", required=True)
parser.add_argument("--cpu-ids", required=True)
parser.add_argument("--warmups", type=int, default=2)
parser.add_argument("--repeats", type=int, default=9)
parser.add_argument("--row-chunk-size", type=int, required=True)
parser.add_argument("--lag-block-size", type=int, required=True)
parser.add_argument("--lag-truncation", type=int, required=True)
parser.add_argument("--theta", type=float, required=True)
parser.add_argument("--phi", type=float, required=True)
parser.add_argument("--reference-output", required=True)
parser.add_argument("--write-reference", action="store_true")
parser.add_argument("--json-output", required=True)
args = parser.parse_args()

cpu_ids = tuple(
    int(token)
    for token in args.cpu_ids.split(",")
    if token.strip()
)
if not cpu_ids:
    raise ValueError("At least one CPU id is required.")

os.sched_setaffinity(0, set(cpu_ids))
n_cpus = len(cpu_ids)

for name in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "TF_NUM_INTRAOP_THREADS",
):
    os.environ[name] = str(n_cpus)
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np

try:
    import psutil
except ImportError:
    psutil = None

from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

from WDM_LISAresponse.code.lisa_response.one_arm_plan import OneArmWDMPlan
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
)


def block_result(value):
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def summarize(values):
    values = np.asarray(values, dtype=float)
    q25, q75 = np.percentile(values, [25.0, 75.0])
    return {
        "median": float(np.median(values)),
        "q25": float(q25),
        "q75": float(q75),
        "iqr": float(q75 - q25),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
        "all": [float(value) for value in values],
    }


with np.load(args.input, allow_pickle=False) as cached:
    times_pad = np.asarray(cached["times_pad"])
    w_plus = np.asarray(cached["w_plus"])
    w_cross = np.asarray(cached["w_cross"])
    Nf = int(cached["Nf"])
    q_wdm = int(cached["q_wdm"])
    d_wdm = int(cached["d_wdm"])
    A_frac = float(cached["A_frac"])
    calc_m0 = bool(cached["calc_m0"])

orbit_data = read_orbit_file(args.orbit_file)
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

grid_context = _build_wdm_dT_grid(
    times_pad,
    Nf=Nf,
    d=d_wdm,
    A_frac=A_frac,
    q=q_wdm,
    calc_m0=calc_m0,
)

shift_kwargs = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "row_chunk_size": int(args.row_chunk_size),
    "lag_block_size": int(args.lag_block_size),
    "batch_chunk": 6,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

build_started = time.perf_counter()
plan = OneArmWDMPlan.build_from_context(
    grid_context,
    orbits,
    float(args.theta),
    float(args.phi),
    L_trunc=int(args.lag_truncation),
    variable_shift_kwargs=shift_kwargs,
)
plan_build_s = time.perf_counter() - build_started

w_plus_device = jnp.asarray(w_plus)
w_cross_device = jnp.asarray(w_cross)
block_result((w_plus_device, w_cross_device))

def run_kernel():
    return plan.apply_batch_device_materialized(
        w_plus_device,
        w_cross_device,
    )

for _ in range(max(0, args.warmups)):
    block_result(run_kernel())

output = block_result(run_kernel())
reference_path = Path(args.reference_output)

if args.write_reference:
    reference_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(reference_path, np.asarray(output))

reference = jnp.asarray(np.load(reference_path, mmap_mode="r"))
difference = output[..., 1:] - reference[..., 1:]
reference_power = jnp.sum(jnp.abs(reference[..., 1:]) ** 2)
difference_power = jnp.sum(jnp.abs(difference) ** 2)
max_abs = jnp.max(jnp.abs(difference))
rel_l2 = jnp.where(
    reference_power > 0,
    jnp.sqrt(difference_power / reference_power),
    jnp.sqrt(difference_power),
)
block_result((max_abs, rel_l2))

wall_times = []
cpu_times = []
effective_cores = []

for _ in range(max(1, args.repeats)):
    cpu_started = time.process_time()
    wall_started = time.perf_counter()
    timed_output = block_result(run_kernel())
    wall_s = time.perf_counter() - wall_started
    cpu_s = time.process_time() - cpu_started

    wall_times.append(wall_s)
    cpu_times.append(cpu_s)
    effective_cores.append(cpu_s / wall_s if wall_s > 0 else float("nan"))

process = psutil.Process(os.getpid()) if psutil is not None else None

result = {
    "platform": platform.platform(),
    "jax_version": jax.__version__,
    "requested_cpu_ids": list(cpu_ids),
    "effective_affinity": sorted(os.sched_getaffinity(0)),
    "allowed_physical_cores": n_cpus,
    "wdm_shape": list(w_plus.shape),
    "output_dtype": str(timed_output.dtype),
    "row_chunk_size": int(args.row_chunk_size),
    "lag_block_size": int(args.lag_block_size),
    "lag_truncation": int(args.lag_truncation),
    "plan_build_s": plan_build_s,
    "wall_seconds": summarize(wall_times),
    "process_cpu_seconds": summarize(cpu_times),
    "effective_core_count": summarize(effective_cores),
    "max_abs_difference": float(max_abs),
    "relative_l2_difference": float(rel_l2),
    "rss_bytes": (
        process.memory_info().rss
        if process is not None
        else None
    ),
    "process_threads": (
        process.num_threads()
        if process is not None
        else None
    ),
}

output_path = Path(args.json_output)
output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(json.dumps(result, indent=2))
"""

WORKER_SCRIPT.write_text(WORKER_SOURCE, encoding="utf-8")
print("Worker:", WORKER_SCRIPT)


## 3. Run the scaling sweep


In [ ]:
scaling_results = []

for index, core_count in enumerate(CORE_COUNTS):
    cpu_ids = PHYSICAL_CPU_IDS[:core_count]
    output_json = NOTEBOOK_DIR / f"_local_wdm_04_cores_{core_count:03d}.json"

    command = [
        sys.executable,
        str(WORKER_SCRIPT),
        "--input", str(INPUT_CACHE),
        "--orbit-file", str(ORBIT_FILE),
        "--cpu-ids", ",".join(str(cpu) for cpu in cpu_ids),
        "--warmups", str(WARMUPS),
        "--repeats", str(REPEATS),
        "--row-chunk-size", str(ROW_CHUNK_SIZE),
        "--lag-block-size", str(LAG_BLOCK_SIZE),
        "--lag-truncation", str(L_TRUNC),
        "--theta", repr(float(theta)),
        "--phi", repr(float(phi)),
        "--reference-output", str(REFERENCE_OUTPUT),
        "--json-output", str(output_json),
    ]
    if index == 0:
        command.append("--write-reference")

    print(f"Running {core_count} physical core(s): {cpu_ids}")
    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(
            f"Worker failed for {core_count} physical cores."
        )

    result = json.loads(output_json.read_text(encoding="utf-8"))
    scaling_results.append(result)
    print(
        f"  median={result['wall_seconds']['median']:.4f} s | "
        f"effective={result['effective_core_count']['median']:.2f} | "
        f"rel_l2={result['relative_l2_difference']:.3e}"
    )


## 4. Summary and plots


In [ ]:
rows = []
one_core_wall = scaling_results[0]["wall_seconds"]["median"]

for result in scaling_results:
    allowed = int(result["allowed_physical_cores"])
    wall = result["wall_seconds"]
    effective = result["effective_core_count"]
    speedup = one_core_wall / wall["median"]

    rows.append(
        {
            "allowed_physical_cores": allowed,
            "median_wall_s": wall["median"],
            "q25_wall_s": wall["q25"],
            "q75_wall_s": wall["q75"],
            "iqr_wall_s": wall["iqr"],
            "speedup_vs_1_core": speedup,
            "parallel_efficiency": speedup / allowed,
            "median_effective_cores": effective["median"],
            "effective_core_fraction": effective["median"] / allowed,
            "relative_l2": result["relative_l2_difference"],
            "rss_gib": (
                result["rss_bytes"] / 1024**3
                if result["rss_bytes"] is not None
                else np.nan
            ),
        }
    )

if pd is not None:
    table = pd.DataFrame(rows)
    display(table)
    table.to_csv(NOTEBOOK_DIR / "_local_wdm_04_cpu_scaling_summary.csv", index=False)
else:
    table = rows
    for row in rows:
        print(row)

(NOTEBOOK_DIR / "_local_wdm_04_cpu_scaling_summary.json").write_text(
    json.dumps(rows, indent=2),
    encoding="utf-8",
)


In [ ]:
import matplotlib.pyplot as plt

core_values = [row["allowed_physical_cores"] for row in rows]
wall_values = [row["median_wall_s"] for row in rows]
speedup_values = [row["speedup_vs_1_core"] for row in rows]
effective_values = [row["median_effective_cores"] for row in rows]

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(core_values, wall_values, marker="o")
ax.set_xlabel("Allowed physical cores")
ax.set_ylabel("Median wall time [s]")
ax.set_title("Production one-arm shift: CPU scaling")
ax.grid(True, alpha=0.3)
fig.savefig(NOTEBOOK_DIR / "_local_wdm_04_wall_time.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(core_values, speedup_values, marker="o", label="Measured")
ax.plot(core_values, core_values, linestyle="--", label="Ideal")
ax.set_xlabel("Allowed physical cores")
ax.set_ylabel("Speedup")
ax.set_title("Production one-arm shift: speedup")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(NOTEBOOK_DIR / "_local_wdm_04_speedup.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(core_values, effective_values, marker="o", label="Measured")
ax.plot(core_values, core_values, linestyle="--", label="Allowed")
ax.set_xlabel("Allowed physical cores")
ax.set_ylabel("Effective cores")
ax.set_title("Process CPU time divided by wall time")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(NOTEBOOK_DIR / "_local_wdm_04_effective_cores.png", dpi=160, bbox_inches="tight")
plt.show()

print("Saved scaling outputs in:", NOTEBOOK_DIR)
